# Moving Average vs Naive Forecast Evaluation

This demo evaluates the forecasting performance of a 3-point moving average versus a naive persistence (last-value) baseline across synthetic noisy time series.

### Key Research Question:
Does smoothing via a moving average reduce mean squared error (MSE) compared to naive persistence under Gaussian noise?

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# numpy and matplotlib are pre-installed on Colab; install locally if needed
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'matplotlib==3.10.0')

### Imports & Setup
Import required libraries for numerical computation and visualization.

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt

# NumPy 2.0 compatibility shims if needed
if not hasattr(np, "alltrue"): np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
if not hasattr(np, "product"): np.product = np.prod

### Data Loading Helper
Load data from GitHub with local fallback to `mini_demo_data.json`.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-0ece95-adaptive-smoothing-and-persistence-trade/main/round-1/experiment-1/demo/mini_demo_data.json"
import os, urllib.request

def load_data():
    try:
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            print("Loaded data from GitHub URL.")
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"Could not load from GitHub ({e}), falling back to local file.")
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            print("Loaded data from local mini_demo_data.json.")
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

data_payload = load_data()

### Configuration
Define tunable experiment parameters.

In [ ]:
# Tunable experiment parameters (minimum viable scale for quick demo)
N_RUNS = 10
SERIES_LENGTH = 20
RANDOM_SEED = 42

### Synthetic Time Series Generation & Forecasting Evaluation
Generate synthetic noisy time series paths and evaluate 3-point Moving Average vs Naive persistence forecasts.

In [ ]:
np.random.seed(RANDOM_SEED)
examples = []

for run in range(N_RUNS):
    t = np.arange(SERIES_LENGTH)
    trend = 0.1 * t
    noise = np.random.normal(0, 1.0, size=SERIES_LENGTH)
    series = trend + noise

    for i in range(3, SERIES_LENGTH):
        ma_pred = float(np.mean(series[i-3:i]))
        naive_pred = float(series[i-1])
        actual = float(series[i])

        examples.append({
            "input": f"Run {run}, step {i}, history: {list(series[i-3:i])}",
            "output": f"{actual}",
            "metadata_run": run,
            "metadata_step": i,
            "predict_moving_average": f"{ma_pred}",
            "predict_naive": f"{naive_pred}"
        })

print(f"Generated {len(examples)} evaluation examples across {N_RUNS} runs.")

### Metric Computation
Compute Mean Squared Error (MSE) for both methods and the performance improvement.

In [ ]:
ma_errors = []
naive_errors = []
for ex in examples:
    actual = float(ex["output"])
    ma_pred = float(ex["predict_moving_average"])
    naive_pred = float(ex["predict_naive"])
    ma_errors.append((actual - ma_pred) ** 2)
    naive_errors.append((actual - naive_pred) ** 2)

ma_mse = float(np.mean(ma_errors))
naive_mse = float(np.mean(naive_errors))
improvement = float(naive_mse - ma_mse)

print(f"Moving Average MSE: {ma_mse:.4f}")
print(f"Naive Persistence MSE: {naive_mse:.4f}")
print(f"Error Reduction (Improvement): {improvement:.4f}")

### Results & Visualization
Visualize a sample time series path along with actual vs predicted values.

In [ ]:
# Visualize one synthetic run
sample_run = 0
t = np.arange(SERIES_LENGTH)
np.random.seed(RANDOM_SEED)
series = 0.1 * t + np.random.normal(0, 1.0, size=SERIES_LENGTH)

steps = []
actuals = []
ma_preds = []
naive_preds = []

for i in range(3, SERIES_LENGTH):
    steps.append(i)
    actuals.append(series[i])
    ma_preds.append(np.mean(series[i-3:i]))
    naive_preds.append(series[i-1])

plt.figure(figsize=(10, 5))
plt.plot(t, series, label='Noisy Series', color='gray', alpha=0.7, marker='o', linestyle='--')
plt.plot(steps, actuals, label='Actual Values', color='black', linewidth=2)
plt.plot(steps, ma_preds, label=f'3-Pt Moving Average (MSE: {ma_mse:.3f})', color='blue', marker='x')
plt.plot(steps, naive_preds, label=f'Naive Persistence (MSE: {naive_mse:.3f})', color='orange', marker='s', linestyle=':')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.title('Time Series Forecasting: Moving Average vs Naive Baseline')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()